In [1]:
# 필요한 패키지
# pip install sqlalchemy pymysql pandas

import pandas as pd
from sqlalchemy import create_engine, text

# ── DB 접속 정보 ───────────────────────────────────────────────────────────────
from DATA.stock_invest_function import get_db_host

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TABLE_NAME = "Korea_company_valuation_ver2"  # 스키마: investar.TABLE_NAME

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# 1) forecast_date의 unique 값 추출
def get_unique_forecast_dates(include_null=False):
    q = f"""
        SELECT DISTINCT forecast_date
        FROM {TABLE_NAME}
        {"WHERE forecast_date IS NOT NULL" if not include_null else ""}
        ORDER BY forecast_date
    """
    with engine.begin() as conn:
        df = pd.read_sql(q, conn, parse_dates=["forecast_date"])
    return df["forecast_date"]

# 2) (ticker, forecast_date, keyword)로 indicator에 keyword가 포함된 값 조회 + date 기준 정렬
#    여러 indicator가 매칭되면 행으로 반환(롱 포맷). wide=True면 indicator별 칼럼으로 피벗.
def get_series_by_keyword(ticker, forecast_date, keyword, wide=False):
    sql = text(f"""
        SELECT `date`, `ticker`, `indicator`, `value`, `forecast_date`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND forecast_date = :fdate
          AND indicator LIKE :kw
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={"ticker": ticker, "fdate": forecast_date, "kw": f"%{keyword}%"},
            parse_dates=["date", "forecast_date"]
        )
    # 숫자형 보정
    if not df.empty:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if wide and not df.empty:
        df_wide = df.pivot_table(index="date", columns="indicator", values="value", aggfunc="last").sort_index()
        df_wide = df_wide.rename_axis(None, axis=1)
        return df_wide
    return df  # 롱 포맷: date, indicator, value …

# 3) indicator의 unique 값 추출
def get_unique_indicators(keyword=None):
    cond = "" if not keyword else "WHERE indicator LIKE :kw"
    sql = text(f"SELECT DISTINCT indicator FROM {TABLE_NAME} {cond} ORDER BY indicator")
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params=(None if not keyword else {"kw": f"%{keyword}%"}))
    return df["indicator"]

# 4) (ticker, indicator, forecast_date 두 개) 입력 시 두 기간 차이 비교
#    반환: date 기준 병합(outer), col: value_fd1, value_fd2, diff = fd2 - fd1
def compare_indicator_between_dates(ticker, indicator, forecast_date_1, forecast_date_2):
    base_sql = text(f"""
        SELECT `date`, `value`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND indicator = :indicator
          AND forecast_date = :fdate
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df1 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_1},
            parse_dates=["date"]
        )
        df2 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_2},
            parse_dates=["date"]
        )

    # 숫자형 보정
    for d in (df1, df2):
        if not d.empty:
            d["value"] = pd.to_numeric(d["value"], errors="coerce")

    df1 = df1.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_1).date()}"})
    df2 = df2.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_2).date()}"})

    out = pd.merge(df1, df2, on="date", how="outer").sort_values("date").set_index("date")
    if out.shape[1] == 2:
        cols = out.columns.tolist()
        out["diff"] = out[cols[1]] - out[cols[0]]  # fd2 - fd1
    return out

# ── 사용 예시 ─────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     # 1) forecast_date 목록
#     print(get_unique_forecast_dates().tail())
#
#     # 2) 키워드로 조회 (롱/와이드)
#     ex_long = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=False)
#     ex_wide = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=True)
#     print(ex_long.head())
#     print(ex_wide.head())
#
#     # 3) indicator 유니크
#     print(get_unique_indicators().head())
#     # 특정 키워드만
#     print(get_unique_indicators(keyword="forecast").head())
#
#     # 4) 두 forecast_date 비교
#     comp = compare_indicator_between_dates(
#         ticker="A005930",
#         indicator="revenue_ensemble_forecast",   # 예: 정확한 indicator 이름 입력
#         forecast_date_1="2025-10-26",
#         forecast_date_2="2025-10-29"
#     )
#     print(comp.tail())



In [2]:
print(get_unique_forecast_dates().tail())

0   2025-10-26
1   2025-10-29
2   2025-10-30
Name: forecast_date, dtype: datetime64[ns]


In [11]:
ex_long = get_series_by_keyword(ticker="A140860", forecast_date="2025-10-30", keyword="mc", wide=False)

In [12]:
ex_long

,date,ticker,indicator,value,forecast_date
0,2025-11-30,A140860,mc_sarima_noexog,2.547052e+09,2025-10-30
1,2025-11-30,A140860,mc_ets,2.665514e+09,2025-10-30
2,2025-11-30,A140860,mc_prophet,2.934204e+09,2025-10-30
3,2025-11-30,A140860,mc_lstm,2.474088e+09,2025-10-30
4,2025-11-30,A140860,mc_theta,2.217052e+09,2025-10-30
...,...,...,...,...,...
60,2026-11-30,A140860,mc_sarima_noexog,2.975848e+09,2025-10-30
61,2026-11-30,A140860,mc_ets,3.109836e+09,2025-10-30
62,2026-11-30,A140860,mc_prophet,2.793635e+09,2025-10-30
63,2026-11-30,A140860,mc_lstm,2.820101e+09,2025-10-30


In [13]:
# ex_long → indicator를 컬럼으로 피벗
ex_pivot = (
    ex_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
ex_pivot.columns.name = None

# 확인
print(ex_pivot.head())


        date   ticker        mc_ets       mc_lstm    mc_prophet  \
0 2025-11-30  A140860  2.665514e+09  2.474088e+09  2.934204e+09   
1 2025-12-31  A140860  2.990192e+09  2.427102e+09  2.977495e+09   
2 2026-01-31  A140860  3.062587e+09  2.440338e+09  3.021079e+09   
3 2026-02-28  A140860  2.288014e+09  2.466731e+09  2.177177e+09   
4 2026-03-31  A140860  2.358973e+09  2.551056e+09  2.188231e+09   

   mc_sarima_noexog      mc_theta  
0      2.547052e+09  2.217052e+09  
1      2.968963e+09  2.484256e+09  
2      2.960907e+09  2.489602e+09  
3      2.258475e+09  2.494947e+09  
4      2.253324e+09  2.430584e+09  


In [14]:
ex_pivot.tail(14)

,date,ticker,mc_ets,mc_lstm,mc_prophet,mc_sarima_noexog,mc_theta
0,2025-11-30,A140860,2.665514e+09,2.474088e+09,2.934204e+09,2.547052e+09,2.217052e+09
1,2025-12-31,A140860,2.990192e+09,2.427102e+09,2.977495e+09,2.968963e+09,2.484256e+09
2,2026-01-31,A140860,3.062587e+09,2.440338e+09,3.021079e+09,2.960907e+09,2.489602e+09
3,2026-02-28,A140860,2.288014e+09,2.466731e+09,2.177177e+09,2.258475e+09,2.494947e+09
4,2026-03-31,A140860,2.358973e+09,2.551056e+09,2.188231e+09,2.253324e+09,2.430584e+09
5,2026-04-30,A140860,2.373126e+09,2.546392e+09,2.178087e+09,2.322264e+09,2.435780e+09
6,2026-05-31,A140860,2.562959e+09,2.539083e+09,2.371063e+09,2.464531e+09,2.440976e+09
7,2026-06-30,A140860,2.786896e+09,2.665901e+09,2.448722e+09,2.707911e+09,2.476452e+09
8,2026-07-31,A140860,2.762988e+09,2.688990e+09,2.421390e+09,2.654423e+09,2.481712e+09
9,2026-08-31,A140860,2.614067e+09,2.707801e+09,2.278124e+09,2.532889e+09,2.486972e+09


In [15]:
psr_long = get_series_by_keyword(ticker="A140860", forecast_date="2025-10-30", keyword="psr", wide=False)

# ex_long → indicator를 컬럼으로 피벗
psr_pivot = (
    psr_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
psr_pivot.columns.name = None

# 확인
print(psr_pivot.head())

        date   ticker       psr  psr_ETS  psr_LSTM  psr_Prophet  \
0 2016-02-29  A140860  4.498726      NaN       NaN          NaN   
1 2016-03-31  A140860  4.801598      NaN       NaN          NaN   
2 2016-04-30  A140860  5.206531      NaN       NaN          NaN   
3 2016-05-31  A140860  6.959583      NaN       NaN          NaN   
4 2016-06-30  A140860  8.285473      NaN       NaN          NaN   

   psr_SARIMA_noexog  psr_Theta  
0                NaN        NaN  
1                NaN        NaN  
2                NaN        NaN  
3                NaN        NaN  
4                NaN        NaN  


In [16]:
psr_pivot.tail(14)

,date,ticker,psr,psr_ETS,psr_LSTM,psr_Prophet,psr_SARIMA_noexog,psr_Theta
116,2025-10-31,A140860,10.080818,NaN,NaN,NaN,NaN,NaN
117,2025-11-30,A140860,NaN,12.097617,11.053034,13.802116,11.685938,10.105045
118,2025-12-31,A140860,NaN,13.275534,11.050931,15.152953,13.305707,10.126833
119,2026-01-31,A140860,NaN,13.596945,11.111197,15.374756,13.269604,10.148622
120,2026-02-28,A140860,NaN,10.158081,11.231367,11.080006,10.121583,10.170410
121,2026-03-31,A140860,NaN,10.333377,11.190989,11.281194,10.031721,10.192199
122,2026-04-30,A140860,NaN,10.395375,11.170529,11.228896,10.338636,10.213987
123,2026-05-31,A140860,NaN,11.226929,11.138466,12.223765,10.972006,10.235776
124,2026-06-30,A140860,NaN,11.782847,11.159311,12.806033,11.688318,10.257564
125,2026-07-31,A140860,NaN,11.681766,11.255960,12.663095,11.457447,10.279353
